In [ ]:
from pathlib import Path

from mff.new_cap import read_extended_base_data
from mff.scenario import calculate_scenario
from mff.scenario_store import deserialize_scenarios

In [ ]:
def find_criss_area_t1(biss_ter):
    if biss_ter <= 1200 and biss_ter != 0:
        return min(10, biss_ter)
    else:
        return 0


def find_criss_area_t2(biss_ter):
    if biss_ter <= 1200 and biss_ter != 0:
        return min(140, max(biss_ter - 10, 0))
    else:
        return 0

In [ ]:
data = read_extended_base_data(2024)

In [ ]:
def calc_young_area(data):
    return data["area_yfs_cur_eligible"].fillna(0.0).clip(lower=0.0, upper=300)

In [ ]:
area1 = data["area_biss_criss"].apply(find_criss_area_t1).sum()
area2 = data["area_biss_criss"].apply(find_criss_area_t2).sum()

res = 189_176_282.78 / (2 * area1 + area2)
redist_per_ha = [2 * res, res]
dabis_per_ha = 735_300_015.55 / data["area_biss_criss"].sum()
yfs_per_ha = 18_690_804.00 / calc_young_area(data).sum()
print(f"Redistributive/ha: {redist_per_ha[0]}, {redist_per_ha[1]} €/ha")
print(f"DABIS/ha: {dabis_per_ha}")
print(f"YFS/ha: {yfs_per_ha}")

In [ ]:
data["area_yfs"].max()

In [ ]:
calc_young_area(data).sum()

In [ ]:
scenarios = deserialize_scenarios(
    Path("data/scenarios.json").read_text(encoding="utf-8")
)

config = scenarios["Ír NegoBox – 2026.10.10. – Status Quo YFS elvonás nélkül"]

result = calculate_scenario(data, config)

print(f"Jelenlegi támogatás: {result.total_current / 1e6:.6f} M€")
print(f"Bruttó DABIS:        {result.total_gross / 1e6:.6f} M€")

print("\nTerületek:")
print(f"Általános: {data['area_biss_criss'].sum():,.2f} ha")
print(f"YFS eredeti: {data['area_yfs_cur_eligible'].sum():,.2f} ha")
print(f"YFS szcenárió: {config.young_area(data).sum():,.2f} ha")

print("\nTámogatási komponensek:")
print(
    f"BISS:  {result.new['area_biss_criss'].sum() * config.dabis_per_ha / 1e6:.6f} M€"
)
print(f"YFS:   {config.young_area(data).sum() * config.yfs_per_ha / 1e6:.6f} M€")
print(
    f"CRISS: {(result.total_gross - result.new['area_biss_criss'].sum() * config.dabis_per_ha - config.young_area(data).sum() * config.yfs_per_ha) / 1e6:.6f} M€"
)